# Mapping the Potential Destructive Power of Wildfires Using Machine Learning
---
## Appendix F: *Raster Processing*
##### Version Number: 3.0
---
### Contents  
> 1. ** 
> 2. **
---
### Notes

- Very early planning stages for hot spot analysis of daily california NDVI raster data
---
### Inputs
- 
---
### Outputs  
- 
---
### User Created Dependencies  
---

In [1]:
import os
import sys

# Allow import of custom modules from the parent directory
sys.path.append(os.path.abspath(os.path.join('..')))

# Show basic facts about a dataframe
from src.data_utils import basic_explore

# basic health checks after a merge
from src.data_utils import post_merge_check

---
### Third Party Dependencies

In [2]:
import pandas as pd
import pygeoprocessing

## Raster Processing Notes

pygeoprocessing.raster_calculator()

pygeoprocessing.raster_calculator(
    base_raster_list,
    local_op,        # function to apply on pixel values
    target_raster_path,
    target_datatype,
    nodata_list=None,
    dtype_mapping=None
)

## Raster Processing Algorithm
0) **Split**
    - Split rasters into each sampling grid area by lat and lon 

1) **Global reference**
    - Compute the global statistic (mean, median, or max and std) to provide context for identifying hotspots/coldspots.

2) **Split raster into overlapping pieces**
    - Overlapping ensures that hotspots near piece edges aren’t missed.
    - n is determined by processor/memory limits.

3) **Assign piece value**
    - Use mean, median, or max and std to summarize each piece.
    - This reduces data resolution for hotspot detection while preserving extremes.

4) **Compare piece values**
    - Use both neighboring pieces and global mean to detect relative extremes.

5) **Label hotspots/coldspots**
    - Each piece is marked as hot, cold, or neutral.

6) **Summarize clusters**
    - Count total hotspots/coldspots.
    - Optionally compute largest cluster size or number of significant clusters for further analysis.

## Hotspot Algorithm

1) **global hot spot**
    - Formula - cell_value > global_mean + k * global_std
2) **local hot spot** 
    - Formula - cell_value > local_mean + k × local_std
3) **Variables**
    - `cell_value` = The value of the raster at the center cell you are currently evaluating.
    - `local_mean` = The average value of all cells in the neighborhood window around the center cell
    - `global_mean` = The average value of all cells in the raster
    - `global_std` = The standard deviation of all the values
    - `local_std` = The standard deviation of the neighborhood values
    - `k` = A multiplier that defines how extreme a value must be to count as a hotspot.
        - Example: k = 2 → more strict: 2 standard deviations above.